In [4]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
from multiprocessing import Process
import multiprocessing
import time
from itertools import product

In [5]:

pkt_prob = 0.5
e_prob = 1
weight_prob = 0.5
#####################################
h_vals = [0.1, 1]
h_bad = 0.5
h_prob = [h_bad, (1-h_bad)]
weight_vals = [1, 2]

B_max = 2
rmax = 1
M = 2

B_vals   = np.arange(0, B_max + 1, 1)
rem_vals = np.arange(0, rmax + 0.5, 0.5)

per_user_space = product(B_vals, rem_vals, h_vals, weight_vals)

# Cartesian product across M users
all_states = list(product(per_user_space, repeat=M))

# --- Reorder to field-wise layout ---
all_states_ordered = []
for state in all_states:
    B   = [user[0] for user in state]
    rem = [user[1] for user in state]
    h   = [user[2] for user in state]
    w   = [user[3] for user in state]
    all_states_ordered.append(tuple(B + rem + h + w))

all_states = np.array(all_states_ordered)

print(all_states.shape)

(1296, 8)


In [6]:
print(all_states[:10])

[[0.  0.  0.  0.  0.1 0.1 1.  1. ]
 [0.  0.  0.  0.  0.1 0.1 1.  2. ]
 [0.  0.  0.  0.  0.1 1.  1.  1. ]
 [0.  0.  0.  0.  0.1 1.  1.  2. ]
 [0.  0.  0.  0.5 0.1 0.1 1.  1. ]
 [0.  0.  0.  0.5 0.1 0.1 1.  2. ]
 [0.  0.  0.  0.5 0.1 1.  1.  1. ]
 [0.  0.  0.  0.5 0.1 1.  1.  2. ]
 [0.  0.  0.  1.  0.1 0.1 1.  1. ]
 [0.  0.  0.  1.  0.1 0.1 1.  2. ]]


In [31]:
P_vals   = np.arange(0, B_max + 1, 1)
rho_vals = np.arange(0, rmax + 0.5, 0.5)

per_user_actions = list(product(P_vals, rho_vals))

all_actions = []
for act in product(per_user_actions, repeat=M):
    P_list   = [user[0] for user in act]
    rho_list = [user[1] for user in act]
    all_actions.append(tuple(P_list + rho_list))

all_actions = np.array(all_actions)
print(all_actions.shape)
# print(all_actions)

(56, 2)


In [32]:
def st_tr_B(B, B_n, P):
    if B - P < 0:
        B_ch = B
    else:
        B_ch = B - P
    if (B_ch == B_max and B_n == B_max):
        m = 1
    else:
        if(B_ch == B_n):
            m = 1 - e_prob
        elif (B_ch+1 == B_n):
            m = e_prob
        else:
            m = 0
    return m
def st_tr_rem (rem_b, rem_b_next, rho):
    # rho = float(rho)
    if (rem_b - rho) < 0:
        rem_ch = rem_b
    else:
        rem_ch = rem_b - rho

    if (rem_ch == rmax and rem_b_next == rmax):
        m = 1
    else:
        if rem_ch == rem_b_next:
            m = 1 - pkt_prob
        elif (rem_b_next == rmax):
            m = pkt_prob
        else:
            m = 0
    return m
def st_tr_wt(wt, wt_next):
    if wt == 1 and wt_next == 1:
        m = (1-pkt_prob) + (pkt_prob) * (weight_prob)
    elif wt == 1 and wt_next == 2:
        m = pkt_prob * (1 - weight_prob)
    elif wt == 2 and wt_next == 1:
        m = pkt_prob * weight_prob
    else:
        m = (1-pkt_prob) + pkt_prob * (1-weight_prob)
    return m

In [33]:
def state_trans_prob(state, next_state, action):
    st_tr_prob = 1.0

    # --- Battery transitions ---
    for i in range(M):
        st_tr_prob *= st_tr_B(state[i], next_state[i], action[i])

    # --- Remaining bits transitions ---
    for i in range(M):
        st_tr_prob *= st_tr_rem(state[M + i], next_state[M + i], action[M + i])

    # --- Channel transitions ---
    for i in range(M):
        idx = h_vals.index(next_state[2*M + i])
        st_tr_prob *= h_prob[idx]

    # --- Weight transitions ---
    for i in range(M):
        st_tr_prob *= st_tr_wt(state[3*M + i], next_state[3*M + i])

    return st_tr_prob

In [34]:
def reward_fn(state, action):
    cost = 0

    # --- Unpack state ---
    B   = state[0:M]
    rem = state[M:2*M]
    h   = state[2*M:3*M]
    wt  = state[3*M:4*M]

    # print(B, rem, h, wt)

    # --- Unpack action ---
    P   = action[0:M]
    rho = action[M:2*M]

    # --- Penalty flag ---
    infeasible = False

    # --- Per-user constraints ---
    for i in range(M):
        mi = np.log2(1 + h[i] * P[i])
        rem_end_i = rem[i] - rho[i]

        if (P[i] > 0 and rho[i] == 0):
            infeasible = True
        elif (P[i] > B[i]):
            infeasible = True
        elif (rho[i] > rem[i]):
            infeasible = True
        elif (rho[i] > mi):
            infeasible = True

        if infeasible:
            cost = 10000
            return cost

    # # --- Full MAC subset constraints ---
    # from itertools import combinations

    # users = list(range(M))
    # for k in range(1, M+1):
    #     for S in combinations(users, k):
    #         sum_rho = sum(rho[i] for i in S)
    #         sum_cap = np.log2(1 + sum(h[i]*P[i] for i in S))
    #         if sum_rho > sum_cap:
    #             cost = 10000
    #             return cost

    # --- Valid action → compute cost ---
    total_cost = 0
    for i in range(M):
        rem_end_i = rem[i] - rho[i]
        total_cost += wt[i] * np.exp(-(rmax - rem_end_i))

    cost = total_cost
    return cost

In [35]:
state = [7, 3, 1, 2]
action = [7, 3]
reward_fn(state, action)

0.09957413673572789

In [36]:
def value_iteration(gamma, j, V_T, send_end):
    act_arr = []
    for act in range(len(all_actions)):
        R_sum = reward_fn(all_states[j], all_actions[act])
        mul1 = 0
        for itr in range(ls):
            v = V_T[itr]
            P_s_tr = state_trans_prob(all_states[j], all_states[itr], all_actions[act])
            mul1 = mul1 + (v*P_s_tr)
        act_arr.append(gamma * mul1 + R_sum)
    fin_v = np.min(act_arr)
    idx = act_arr.index(fin_v)
    temp = [fin_v, idx]
    send_end.send(temp)

In [37]:
T = 5
ls = len(all_states)
V_T = np.zeros(ls)
Ix_T = np.zeros(ls)
gamma = 0.99
stp = 0
while(stp < T):
    # t1 = time.time()
    print('itr', stp)
    q = 0
    z = 28
    V_Ttemp = np.array([])
    Ix_Ttemp = np.array([])
    while z <= len(all_states):
        jobs = []
        pipe_list = []
        for j in range(q, z):
            recv_end, send_end = multiprocessing.Pipe(False)
            p = Process(target=value_iteration, args = (gamma, j, V_T, send_end))
            jobs.append(p)
            pipe_list.append(recv_end)
        for process in jobs:
            process.start()
        for process in jobs:
            process.join()
        temp2 = np.array([x.recv() for x in pipe_list])
        V_Ttemp = np.concatenate((V_Ttemp, temp2[:, 0]))
        Ix_Ttemp = np.concatenate((Ix_Ttemp, temp2[:, 1]))
        q = z
        z+= 28
    V_T = V_Ttemp
    # print('time per itr', time.time() - t1)
    stp+= 1
print('value iteration done', e_prob, pkt_prob, weight_prob)

itr 0
itr 1
itr 2
itr 3
itr 4
value iteration done 1 0.5 0.5


In [38]:
Ix_T = Ix_Ttemp

In [39]:
for i in range(len(all_states)):
    idx = int(Ix_T[i])
    act = all_actions[idx]
    print(all_states[i],'--',act)

[0.  0.  0.1 1. ] -- [0. 0.]
[0.  0.  0.1 2. ] -- [0. 0.]
[0. 0. 1. 1.] -- [0. 0.]
[0. 0. 1. 2.] -- [0. 0.]
[0.  0.5 0.1 1. ] -- [0. 0.]
[0.  0.5 0.1 2. ] -- [0. 0.]
[0.  0.5 1.  1. ] -- [0. 0.]
[0.  0.5 1.  2. ] -- [0. 0.]
[0.  1.  0.1 1. ] -- [0. 0.]
[0.  1.  0.1 2. ] -- [0. 0.]
[0. 1. 1. 1.] -- [0. 0.]
[0. 1. 1. 2.] -- [0. 0.]
[0.  1.5 0.1 1. ] -- [0. 0.]
[0.  1.5 0.1 2. ] -- [0. 0.]
[0.  1.5 1.  1. ] -- [0. 0.]
[0.  1.5 1.  2. ] -- [0. 0.]
[0.  2.  0.1 1. ] -- [0. 0.]
[0.  2.  0.1 2. ] -- [0. 0.]
[0. 2. 1. 1.] -- [0. 0.]
[0. 2. 1. 2.] -- [0. 0.]
[0.  2.5 0.1 1. ] -- [0. 0.]
[0.  2.5 0.1 2. ] -- [0. 0.]
[0.  2.5 1.  1. ] -- [0. 0.]
[0.  2.5 1.  2. ] -- [0. 0.]
[0.  3.  0.1 1. ] -- [0. 0.]
[0.  3.  0.1 2. ] -- [0. 0.]
[0. 3. 1. 1.] -- [0. 0.]
[0. 3. 1. 2.] -- [0. 0.]
[1.  0.  0.1 1. ] -- [0. 0.]
[1.  0.  0.1 2. ] -- [0. 0.]
[1. 0. 1. 1.] -- [0. 0.]
[1. 0. 1. 2.] -- [0. 0.]
[1.  0.5 0.1 1. ] -- [0. 0.]
[1.  0.5 0.1 2. ] -- [0. 0.]
[1.  0.5 1.  1. ] -- [0. 0.]
[1.  0.5 1.  2. ] -- [0. 

In [40]:
# M = 2
# T_horizon = 100000

# tot_dist = 0
# state_slot = [0]*M + [0]*M + [h_vals[0]]*M + [1]*M   # initial state
# wt_next = [1]*M

# import time
# t_cal = 0

# for t in range(T_horizon):
#     t1 = time.time()

#     s_idx = all_states.tolist().index(state_slot)
#     idx = int(Ix_T[s_idx])
#     act = all_actions[idx]

#     # --- Unpack action ---
#     P   = act[0:M]
#     rho = act[M:2*M]

#     # --- Unpack state ---
#     B   = state_slot[0:M]
#     rem = state_slot[M:2*M]

#     # --- Distortion / cost ---
#     dist = 0
#     for i in range(M):
#         dist += wt_next[i] * np.exp(-(rmax - (rem[i] - rho[i])))

#     # --- Random realizations ---
#     wt  = np.random.choice([1, 2], size=M, p=[weight_prob, 1-weight_prob])
#     h   = np.random.choice(h_vals, size=M, p=h_prob)
#     pkt = np.random.choice([1, 0], size=M, p=[pkt_prob, 1-pkt_prob])
#     E   = np.random.choice([1, 0], size=M, p=[e_prob, 1-e_prob])

#     # --- Next battery ---
#     next_B = [min(B[i] - P[i] + E[i], B_max) for i in range(M)]

#     # --- Next remaining bits & weights ---
#     next_bits = [0]*M
#     for i in range(M):
#         if pkt[i] == 1:
#             next_bits[i] = rmax
#             wt_next[i] = wt[i]
#         else:
#             next_bits[i] = rem[i] - rho[i]

#     # --- Next state ---
#     next_state = (
#         next_B +
#         next_bits +
#         list(h) +
#         wt_next
#     )

#     tot_dist += dist
#     state_slot = next_state

#     if t % 10000 == 0:
#         print(t, tot_dist/((t+1)*M))

#     t_cal += time.time() - t1

# fin_result = tot_dist/(T_horizon*M)
# print('final objective', fin_result)
# print('-------------------------------------')